https://github.com/<redacted>/assignment-4-equation-of-a-slime-<redacted>

763da15

In [1]:
# Imports section
import pandas
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from random import uniform

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset
df = pandas.read_csv("science_data_large.csv")
# Output the first 15 rows of the data
print(df.head(15))
# Display a summary of the table information (number of datapoints, etc.)
print(df.info())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[["Temperature °C", "Mols KCL"]]
y = df["Size nm^3"]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## Part 3. Perform a Linear Regression

In [4]:
# Use sklearn to train a model on the training set

model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model

sample_datapoint = pandas.DataFrame({"Temperature °C": [246], "Mols KCL": [601]})
prediction = model.predict(sample_datapoint)
print("predicted size:")
print(prediction[0])

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print("r² score:")
print(model.score(X_test, y_test))
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print("coefficients:")
print(model.coef_)
print("intercept:")
print(model.intercept_)
print("testing:")
print(246 * model.coef_[0] + 601 * model.coef_[1] + model.intercept_)

predicted size:
424330.27306404815
r² score:
0.8552472077276096
coefficients:
[ 866.14641337 1032.69506649]
intercept:
-409391.4795834081
testing:
424330.27306404815


Our model predicts that with a temperature of aproximately 246°C and 602 mols of KCL, the slime will have a size of 424330.27306404815 nm³.

Our r² score indicates that aproximately 86% of the variance in the output can be explained by the variance in the input.

Equation: $h(x)=866.14641337t+1032.69506649k$

Where t is temperature in celsius and k is the number of moles of potassium chloride.

## Part 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data

scores = cross_val_score(model, X, y, cv = 5, scoring = "r2")

# Report on their finding and their significance
print("scores:")
print(scores)
print("average score:")
print(scores.mean())

scores:
[0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
average score:
0.8568167899144437


On average, aproximately 86% of the variance in our model's output can be explained by the variance in the input.

## Part 5. Using Polynomial Regression

In [6]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2

#"augmenting" the dataset
augmented_samples = []
for _, row in df.iterrows():
    temp = row["Temperature °C"] + uniform(-min(100, row["Temperature °C"]), 100)
    kcl = row["Mols KCL"] + uniform(-min(100, row["Mols KCL"]), 100)
    size = row["Size nm^3"] + uniform(-min(100000, row["Mols KCL"]), 100000)
    augmented_samples.append((row["Temperature °C"], row["Mols KCL"], row["Size nm^3"]))
    augmented_samples.append((temp, kcl, size))

augmented_df = pandas.DataFrame(augmented_samples, columns=['temperature', 'KCL', 'size'])

X_aug = augmented_df[['temperature', 'KCL']]
y_aug = augmented_df['size']

X_aug_train, X_aug_test, y_aug_train, y_aug_test = train_test_split(X, y, test_size=0.1, random_state=42)

poly_features = PolynomialFeatures(degree = 2)
X_train_poly = poly_features.fit_transform(X_aug_train)
X_test_poly = poly_features.transform(X_aug_test)

model = LinearRegression()
model.fit(X_train_poly, y_aug_train)
# Report on the metrics and output the resultant equation as you did in Part 3.
print("score")
print(model.score(X_test_poly, y_aug_test))
coefficients = model.coef_
print("coefficients:")
print(coefficients)
print("intercept:")
print(model.intercept_)


score
1.0
coefficients:
[ 0.00000000e+00  1.20000000e+01 -1.27198206e-07  1.26473309e-11
  2.00000000e+00  2.85714287e-02]
intercept:
2.047908492386341e-05


It seems that 100% of the variance in our model's output can be explained by the variance in the input.

Equation: $h(x)=0.00002047908492386341+12t-0.000000127198206k+0.0000000000126473309t^2+2tk+0.0285714287k^2$

Where t is temperature in celsius and k is the number of moles of potassium chloride.